In [1]:
import sqlite3
import pandas as pd

In [16]:

# 1. OPEN DATABASE CONNECTION


db_name = "sofia_macro_finance.db"

conn = sqlite3.connect(db_name)

# Enable foreign-key validation in SQLite. PRAGMA is SQLite command
conn.execute("PRAGMA foreign_keys = ON")

cursor = conn.cursor()

print(f"Successfully connected to database: {db_name}")


Successfully connected to database: sofia_macro_finance.db


In [17]:

try:
    
    # 2. CREATE DATABASE TABLES
    

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS macro_indicators (
            record_id INTEGER PRIMARY KEY AUTOINCREMENT,
            country_code TEXT NOT NULL,
            date TEXT NOT NULL,
            gdp_growth_pct REAL,
            cpi_inflation_pct REAL,
            central_bank_rate REAL,

            UNIQUE (country_code, date)
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS corporate_financials (
            statement_id INTEGER PRIMARY KEY AUTOINCREMENT,
            ticker TEXT NOT NULL,
            country_code TEXT NOT NULL,
            statement_date TEXT NOT NULL,
            fiscal_year INTEGER NOT NULL,
            fiscal_quarter INTEGER NOT NULL,
            revenue_usd REAL,
            net_income_usd REAL,
            total_assets_usd REAL,

            FOREIGN KEY (
                country_code,
                statement_date
            )
            REFERENCES macro_indicators (
                country_code,
                date
            ),

            UNIQUE (
                ticker,
                fiscal_year,
                fiscal_quarter
            )
        )
    """)

    conn.commit()

    print("Database tables created successfully.")


    # =========================================================================
    # 3. PREPARE SAMPLE MACROECONOMIC DATA
    # =========================================================================

    macro_data = [
        (
            "USA",
            "2025-12-31",
            2.4,
            3.1,
            4.5
        ),
        (
            "USA",
            "2026-03-31",
            2.1,
            2.9,
            4.2
        ),
        (
            "EMU",
            "2025-12-31",
            1.1,
            2.2,
            3.0
        ),
        (
            "USA",
            "2026-04-30",
            2.5,
            3.0,
            4.5
        )
            ]


    # =========================================================================
    # 4. INSERT MACROECONOMIC DATA
    # =========================================================================

    cursor.executemany("""
        INSERT OR IGNORE INTO macro_indicators (
            country_code,
            date,
            gdp_growth_pct,
            cpi_inflation_pct,
            central_bank_rate
        )
        VALUES (?, ?, ?, ?, ?)
    """, macro_data)


    
    # 5. SAMPLE CORPORATE DATA
    

    corporate_data = [
        (
            "AAPL",
            "USA",
            "2025-12-31",
            2025,
            4,
            94_930_000_000.0,
            21_440_000_000.0,
            365_000_000_000.0
        ),
        (
            "AAPL",
            "USA",
            "2026-03-31",
            2026,
            1,
            111_200_000_000.0,
            29_100_000_000.0,
            380_000_000_000.0
        ),
        (
            "ASML",
            "EMU",
            "2025-12-31",
            2025,
            4,
            7_500_000_000.0,
            1_900_000_000.0,
            42_000_000_000.0
        )
    ]


    # =========================================================================
    # 6. INSERT CORPORATE DATA
    # =========================================================================

    cursor.executemany("""
        INSERT OR IGNORE INTO corporate_financials (
            ticker,
            country_code,
            statement_date,
            fiscal_year,
            fiscal_quarter,
            revenue_usd,
            net_income_usd,
            total_assets_usd
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, corporate_data)

    conn.commit()

    print("Sample data inserted successfully.")


    
    # 7. DEFINE QUERIES
    

    target_ticker = "AAPL"

    sql_query = """
        SELECT
            f.ticker,
            f.statement_date,
            f.fiscal_year,
            f.fiscal_quarter,
            f.revenue_usd,
            f.net_income_usd,
            f.total_assets_usd,
            m.country_code,
            m.gdp_growth_pct
                AS national_gdp_growth,
            m.cpi_inflation_pct
                AS national_inflation,
            m.central_bank_rate
                AS national_central_bank_rate
        FROM corporate_financials AS f
        JOIN macro_indicators AS m
            ON f.country_code = m.country_code
           AND f.statement_date = m.date
        WHERE f.ticker = ?
        ORDER BY
            f.fiscal_year,
            f.fiscal_quarter
    """
    profitability_query = """
        SELECT
            ticker,
            fiscal_year,
            fiscal_quarter,
            revenue_usd,
            net_income_usd,
            ROUND(
                net_income_usd /
                NULLIF(revenue_usd, 0) * 100,
                2
            ) AS net_profit_margin_pct
        FROM corporate_financials
        ORDER BY
            ticker,
            fiscal_year,
            fiscal_quarter
    """
  

    country_summary_query = """
        SELECT
            m.country_code,
            COUNT(f.statement_id)
                AS statement_count,
            SUM(f.revenue_usd)
                AS total_revenue_usd,
            ROUND(
                AVG(f.net_income_usd),
                2
            ) AS average_net_income_usd
        FROM macro_indicators AS m
        JOIN corporate_financials AS f
            ON m.country_code = f.country_code
           AND m.date = f.statement_date
        GROUP BY
            m.country_code            
        ORDER BY
            total_revenue_usd
            
    """

    above_country_average_query = """
        SELECT
            f1.ticker,
            f1.country_code,
            f1.fiscal_year,
            f1.fiscal_quarter,
            f1.revenue_usd
        FROM corporate_financials AS f1
        WHERE f1.revenue_usd > (
            SELECT AVG(f2.revenue_usd)
            FROM corporate_financials AS f2
            WHERE f2.country_code = f1.country_code
        )
        ORDER BY
            f1.country_code,
            f1.revenue_usd DESC
    """
    
    # 8. EXECUTE THE QUERIES AND CREATE A DATAFRAMES
    

    blended_df = pd.read_sql_query(
        sql_query,
        conn,
        params=(target_ticker,)
    )
    profitability_df = pd.read_sql_query(
        profitability_query,
        conn
    )
    country_summary_df = pd.read_sql_query(
    country_summary_query,
    conn
    )

    above_country_average_df = pd.read_sql_query(
    above_country_average_query,
    conn
)



    # =========================================================================
    # 9. DISPLAY THE RESULT
    # =========================================================================

    print("\nCorporate data combined with macroeconomic data:\n")
    print(blended_df)

    print("\nCorporate profitability:\n")
    print(profitability_df)

    print("\nFinancial summary by country and period:\n")
    print(country_summary_df)

    print("\nStatements with revenue above their country average:\n")
    print(above_country_average_df)


# =============================================================================
# 10. CLOSE THE DATABASE CONNECTION
# =============================================================================

finally:
    conn.close()
    print("\nDatabase connection closed.")

Database tables created successfully.
Sample data inserted successfully.

Corporate data combined with macroeconomic data:

  ticker statement_date  fiscal_year  fiscal_quarter   revenue_usd  \
0   AAPL     2025-12-31         2025               4  9.493000e+10   
1   AAPL     2026-03-31         2026               1  1.112000e+11   

   net_income_usd  total_assets_usd country_code  national_gdp_growth  \
0    2.144000e+10      3.650000e+11          USA                  2.4   
1    2.910000e+10      3.800000e+11          USA                  2.1   

   national_inflation  national_central_bank_rate  
0                 3.1                         4.5  
1                 2.9                         4.2  

Corporate profitability:

  ticker  fiscal_year  fiscal_quarter   revenue_usd  net_income_usd  \
0   AAPL         2025               4  9.493000e+10    2.144000e+10   
1   AAPL         2026               1  1.112000e+11    2.910000e+10   
2   ASML         2025               4  7.500000e+